In [18]:
"""
=============================================================
 RETAIL DEMAND FORECASTING — FULL ML PIPELINE
 Challenge: Predict next 2 weeks daily demand
=============================================================
 USAGE:
   1. Place train.csv, test.csv, sample_submission.csv
      in the same folder as this script.
   2. pip install pandas numpy scikit-learn catboost xgboost optuna
   3. python demand_forecasting_solution.py
=============================================================
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
from catboost import CatBoostRegressor

# ─────────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────────
SEED = 42
N_SPLITS = 5
DATA_DIR = Path(".")          # folder with CSVs
OUT_PATH  = DATA_DIR / "submission.csv"

np.random.seed(SEED)

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data …")
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test  = pd.read_csv(DATA_DIR / "test.csv",  parse_dates=["date"])
sub   = pd.read_csv(DATA_DIR / "ans.csv")

# ─────────────────────────────────────────────
# 2. COMBINE FOR FEATURE ENGINEERING
# ─────────────────────────────────────────────
train["_split"] = "train"
test["_split"]  = "test"
test["demand"]  = np.nan

df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)

# ─────────────────────────────────────────────
# 3. TEMPORAL FEATURES
# ─────────────────────────────────────────────
print("Building temporal features …")
df["year"]        = df["date"].dt.year
df["month"]       = df["date"].dt.month
df["day"]         = df["date"].dt.day
df["dayofweek"]   = df["date"].dt.dayofweek   # 0=Mon, 6=Sun
df["weekofyear"]  = df["date"].dt.isocalendar().week.astype(int)
df["quarter"]     = df["date"].dt.quarter
df["is_weekend"]  = (df["dayofweek"] >= 5).astype(int)

# Proximity to holiday (days before/after nearest holiday)
holiday_dates = df.loc[df["is_holiday"] == 1, "date"].unique()
def days_to_nearest_holiday(d):
    if len(holiday_dates) == 0:
        return 0
    diffs = np.abs((holiday_dates - d).astype("timedelta64[s]").astype(int))
    return diffs.min()

unique_dates = df["date"].unique()
holiday_proximity = {d: days_to_nearest_holiday(d) for d in unique_dates}
df["days_to_holiday"] = df["date"].map(holiday_proximity)

# ─────────────────────────────────────────────
# 4. LAG & ROLLING FEATURES  (only from train)
# ─────────────────────────────────────────────
print("Building lag/rolling features …")

# We compute lags on the FULL sorted series; test rows have NaN demand,
# so lags reference only actual past observations automatically.
grp = df.groupby(["store_id", "product_id"])["demand"]

for lag in [7, 14, 21, 28]:
    df[f"lag_{lag}"]  = grp.shift(lag)

for window in [7, 14, 28]:
    shifted = df.groupby(["store_id", "product_id"])["demand"].shift(7)
    rolled  = shifted.groupby([df["store_id"], df["product_id"]]) \
                     .rolling(window, min_periods=1)
    df[f"roll_mean_{window}"] = rolled.mean().droplevel([0, 1])
    df[f"roll_std_{window}"]  = rolled.std().droplevel([0, 1])
    df[f"roll_max_{window}"]  = rolled.max().droplevel([0, 1])
    df[f"roll_min_{window}"]  = rolled.min().droplevel([0, 1])

# Sell-through trend: ratio of last 7 days mean vs last 28 days mean
df["sell_through_trend"] = (
    df["roll_mean_7"] / (df["roll_mean_28"] + 1e-6)
)

# Same weekday last week / 2 weeks ago
df["lag_same_dow_1w"] = grp.shift(7)
df["lag_same_dow_2w"] = grp.shift(14)

# Expanding mean (overall product-store history)
df["expanding_mean"] = (
    grp.shift(1)
      .expanding()
      .mean()
      .reset_index(level=0, drop=True)
)

# ─────────────────────────────────────────────
# 5. PRICE FEATURES
# ─────────────────────────────────────────────
print("Building price features …")

# Price relative to store-level median for same product
store_prod_median = (
    train.groupby(["store_id", "product_id"])["price"].median()
      .rename("price_store_median")
)
df = df.join(store_prod_median, on=["store_id", "product_id"])
df["price_rel_store"] = df["price"] / (df["price_store_median"] + 1e-6)

# Price relative to city-level median for same product (proxy for competition)
city_prod_median = (
    train.groupby(["city", "product_id"])["price"].median()
      .rename("price_city_median")
)
df = df.join(city_prod_median, on=["city", "product_id"])
df["price_rel_city"]  = df["price"] / (df["price_city_median"] + 1e-6)

# Price change from prior day
df["price_lag1"]   = grp_price = df.groupby(["store_id","product_id"])["price"].shift(1)
df["price_delta"]  = df["price"] - df["price_lag1"]
df["price_pct_chg"]= df["price_delta"] / (df["price_lag1"] + 1e-6)

# ─────────────────────────────────────────────
# 6. SPATIAL FEATURES
# ─────────────────────────────────────────────
print("Building spatial features …")

# Store-level aggregate demand statistics
store_stats = (
    train.groupby("store_id")["demand"]
      .agg(store_demand_mean="mean", store_demand_std="std")
)
df = df.join(store_stats, on="store_id")

# City-level product popularity rank
city_prod_rank = (
    train.groupby(["city","product_id"])["demand"].mean()
      .groupby(level=0).rank(ascending=False)
      .rename("city_prod_rank")
)
df = df.join(city_prod_rank, on=["city","product_id"])

# Region-level demand index
region_mean = (
    train.groupby("region")["demand"].mean().rename("region_demand_mean")
)
df = df.join(region_mean, on="region")

# ─────────────────────────────────────────────
# 7. NEW PRODUCT HANDLING
# ─────────────────────────────────────────────
print("Handling new products …")

# Compute days since product first appeared in each store
first_seen = (
    train.groupby(["store_id","product_id"])["date"].min()
      .rename("first_seen_date")
)
df = df.join(first_seen, on=["store_id","product_id"])
df["product_age_days"] = (df["date"] - df["first_seen_date"]).dt.days.clip(lower=0)
df["is_new_product"]   = (df["product_age_days"] < 30).astype(int)

# ─────────────────────────────────────────────
# 8. ENCODE CATEGORICALS
# ─────────────────────────────────────────────
print("Encoding categoricals …")

cat_cols = ["store_id", "product_id", "city", "region",
            "product_name", "category", "holiday_name"]

for col in cat_cols:
    df[col] = df[col].astype(str).fillna("MISSING")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# ─────────────────────────────────────────────
# 9. DEFINE FEATURE SET
# ─────────────────────────────────────────────
DROP_COLS = ["row_ID", "date", "demand", "_split",
             "price_lag1", "first_seen_date", "price_store_median",
             "price_city_median"]

FEATURES = [c for c in df.columns if c not in DROP_COLS]

print(f"\n→ Total features: {len(FEATURES)}")
print(FEATURES)

# ─────────────────────────────────────────────
# 10. SPLIT BACK
# ─────────────────────────────────────────────
tr = df[df["_split"] == "train"].copy()
te = df[df["_split"] == "test"].copy()

X_train = tr[FEATURES]
y_train = tr["demand"].clip(lower=0)   # demand can't be negative

X_test  = te[FEATURES]

# ─────────────────────────────────────────────
# 11. CROSS-VALIDATION STRATEGY
#     GroupKFold on week number → no data leakage
# ─────────────────────────────────────────────
tr["week_group"] = (
    (tr["date"] - tr["date"].min()).dt.days // 7
)
gkf = GroupKFold(n_splits=N_SPLITS)
cv_groups = tr["week_group"]

# ─────────────────────────────────────────────
# 12. CATBOOST MODEL
# ─────────────────────────────────────────────
print("\n── Training CatBoost ──")

cb_params = {
    "loss_function":     "RMSE",
    "iterations":        3000,
    "learning_rate":     0.03,
    "depth":             5,
    "l2_leaf_reg":       3,
    "subsample":         0.8,
    "colsample_bylevel": 0.8,
    "min_data_in_leaf":  30,
    "random_seed":       SEED,
    "thread_count":      -1,
    "verbose":           False,
    "early_stopping_rounds": 100,
}

oof_cb  = np.zeros(len(tr))
pred_cb = np.zeros(len(te))

for fold, (idx_tr, idx_val) in enumerate(
        gkf.split(X_train, y_train, groups=cv_groups), 1):

    Xtr, ytr = X_train.iloc[idx_tr], y_train.iloc[idx_tr]
    Xval, yval = X_train.iloc[idx_val], y_train.iloc[idx_val]

    model = CatBoostRegressor(**cb_params)
    model.fit(
        Xtr, ytr,
        eval_set=(Xval, yval),
    )

    oof_cb[idx_val] = model.predict(Xval).clip(min=0)
    pred_cb        += model.predict(X_test).clip(min=0) / N_SPLITS

    mae = mean_absolute_error(yval, oof_cb[idx_val])
    print(f"  Fold {fold} — MAE: {mae:.4f} | best iter: {model.best_iteration_}")

cb_oof_mae = mean_absolute_error(y_train, oof_cb)
print(f"\n  CatBoost OOF MAE: {cb_oof_mae:.4f}")

# ─────────────────────────────────────────────
# 13. XGBOOST MODEL  (for ensemble diversity)
# ─────────────────────────────────────────────


Loading data …
Building temporal features …
Building lag/rolling features …
Building price features …
Building spatial features …
Handling new products …
Encoding categoricals …

→ Total features: 49
['row_id', 'store_id', 'city', 'region', 'product_id', 'product_name', 'category', 'price', 'loyalty_day', 'is_holiday', 'holiday_name', 'year', 'month', 'day', 'dayofweek', 'weekofyear', 'quarter', 'is_weekend', 'days_to_holiday', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_max_7', 'roll_min_7', 'roll_mean_14', 'roll_std_14', 'roll_max_14', 'roll_min_14', 'roll_mean_28', 'roll_std_28', 'roll_max_28', 'roll_min_28', 'sell_through_trend', 'lag_same_dow_1w', 'lag_same_dow_2w', 'expanding_mean', 'price_rel_store', 'price_rel_city', 'price_delta', 'price_pct_chg', 'store_demand_mean', 'store_demand_std', 'city_prod_rank', 'region_demand_mean', 'product_age_days', 'is_new_product']

── Training CatBoost ──
  Fold 1 — MAE: 1.0857 | best iter: 784
  Fold 2 — MAE: 1.0

In [20]:
te

,row_id,date,store_id,city,region,product_id,product_name,category,price,loyalty_day,...,price_lag1,price_delta,price_pct_chg,store_demand_mean,store_demand_std,city_prod_rank,region_demand_mean,first_seen_date,product_age_days,is_new_product
726,0,2025-05-11,0,0,2,0,2,0,4.08,1,...,4.08,0.0,0.0,1.213974,2.305168,1.0,1.08382,2023-05-15,727,0
727,580,2025-05-12,0,0,2,0,2,0,4.08,0,...,4.08,0.0,0.0,1.213974,2.305168,1.0,1.08382,2023-05-15,728,0
728,1160,2025-05-13,0,0,2,0,2,0,4.08,1,...,4.08,0.0,0.0,1.213974,2.305168,1.0,1.08382,2023-05-15,729,0
729,1740,2025-05-14,0,0,2,0,2,0,4.08,1,...,4.08,0.0,0.0,1.213974,2.305168,1.0,1.08382,2023-05-15,730,0
730,2320,2025-05-15,0,0,2,0,2,0,4.08,1,...,4.08,0.0,0.0,1.213974,2.305168,1.0,1.08382,2023-05-15,731,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429195,5799,2025-05-20,9,9,0,57,16,4,15.11,0,...,15.11,0.0,0.0,1.108150,2.120100,54.0,1.10815,2023-05-15,736,0
429196,6379,2025-05-21,9,9,0,57,16,4,15.11,0,...,15.11,0.0,0.0,1.108150,2.120100,54.0,1.10815,2023-05-15,737,0
429197,6959,2025-05-22,9,9,0,57,16,4,15.11,0,...,15.11,0.0,0.0,1.108150,2.120100,54.0,1.10815,2023-05-15,738,0
429198,7539,2025-05-23,9,9,0,57,16,4,15.11,0,...,15.11,0.0,0.0,1.108150,2.120100,54.0,1.10815,2023-05-15,739,0


In [ ]:
print(f"\n── Writing submission to {OUT_PATH} ──")

te_result = te[["row_id"]].copy()
te_result["demand"] = final_preds

# Merge with sample_submission to preserve ordering
sub_out = sub[["row_id"]].merge(te_result, on="row_id", how="left")
sub_out["demand"] = sub_out["demand"].fillna(0).clip(lower=0)

sub_out.to_csv(OUT_PATH, index=False)
print(f"  Done! Shape: {sub_out.shape}")
print(sub_out.head(10))

# ─────────────────────────────────────────────
# 16. FEATURE IMPORTANCE  (CatBoost)
# ─────────────────────────────────────────────
print("\n── Top 20 features (CatBoost) ──")
# Re-train on all data for importance display
final_cb = CatBoostRegressor(**{**cb_params, "iterations": 2000})
final_cb.fit(X_train, y_train)

importance = pd.Series(
    final_cb.get_feature_importance(), index=FEATURES
).sort_values(ascending=False)

print(importance.head(20).to_string())
print("\n✅ Pipeline complete.")


── Writing submission to submission.csv ──
  Done! Shape: (8120, 2)
   row_id    demand
0       0  5.695268
1       1  2.430346
2       2  0.975776
3       3  0.048972
4       4  0.222671
5       5  0.050013
6       6  0.117788
7       7  2.134400
8       8  1.718189
9       9  0.610012

── Top 20 features (CatBoost) ──
roll_mean_28          21.654684
roll_mean_14          13.243405
roll_mean_7            9.949089
city_prod_rank         9.075504
lag_7                  6.720444
roll_min_7             6.284431
lag_same_dow_1w        4.932220
price                  4.909486
roll_min_14            2.507342
dayofweek              1.897795
lag_same_dow_2w        1.860521
product_name           1.488499
lag_14                 1.420383
sell_through_trend     1.371205
roll_std_28            1.194016
lag_28                 1.176225
product_id             1.163530
store_demand_mean      0.713236
lag_21                 0.697382
roll_max_14            0.694038

✅ Pipeline complete.
